In [16]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [17]:
poi_df = pd.read_csv("data/station_pois.csv")
crime_df = pd.read_csv("data/crime_with_weather.csv")
stations = gpd.GeoDataFrame(
    {
        "name": [
            "9th Street","7th Street","CTC/Arena","3rd St/Convention",
            "Brooklyn Village","Carson","Bland","East/West"
        ],
        "lat": [35.22970,35.22722,35.22500,35.22361,35.22139,35.21889,35.21583,35.21194],
        "lon": [-80.83500,-80.83806,-80.84139,-80.84306,-80.84694,-80.85083,-80.85528,-80.85917]
    },
    geometry=[Point(xy) for xy in zip([-80.83500,-80.83806,-80.84139,-80.84306,-80.84694,-80.85083,-80.85528,-80.85917],
                                      [35.22970,35.22722,35.22500,35.22361,35.22139,35.21889,35.21583,35.21194])],
    crs="EPSG:4326"
)
stations_walkshed_4326 = gpd.read_file("data/station_walksheds.geojson")

In [18]:
def assign_station_lists(points_gdf: gpd.GeoDataFrame,
                         stations_points_4326: gpd.GeoDataFrame,
                         walksheds_4326: gpd.GeoDataFrame,
                         station_name_col: str = "name",
                         out_col: str = "stations_in_radius",
                         out_dist_col: str = "stations_in_radius_dist_m",
                         crs_meters: str = "EPSG:26917") -> gpd.GeoDataFrame:
    """
    For each point, find ALL station walksheds that contain it (within),
    then return ordered station name list (closest->farthest) by distance
    to the station POINT geometry (in meters).
    """

    # Keep a stable id for grouping (index can get messy after joins)
    pts = points_gdf.copy()
    pts["_pt_id"] = range(len(pts))

    # Project to a meters CRS for distance calcs
    pts_m = pts.to_crs(crs_meters)
    walksheds_m = walksheds_4326[[station_name_col, "geometry"]].to_crs(crs_meters)
    stations_m = stations_points_4326[[station_name_col, "geometry"]].to_crs(crs_meters)

    # Spatial join: each point may match multiple walksheds
    joined = gpd.sjoin(
        pts_m[["_pt_id", "geometry"]],
        walksheds_m,
        predicate="within",
        how="left"
    ).rename(columns={station_name_col: "_station"})

    # Attach the station POINT geometry so we can compute point->station distance
    stations_m2 = stations_m.rename(columns={"geometry": "_station_geom"})
    joined = joined.merge(
        stations_m2[[station_name_col, "_station_geom"]].rename(columns={station_name_col: "_station"}),
        on="_station",
        how="left"
    )

    # Distance in meters (NaN where no station matched)
    joined["_dist_m"] = joined.geometry.distance(joined["_station_geom"])

    # Build ordered lists per point
    def agg_station_lists(df):
        df = df.dropna(subset=["_station", "_dist_m"]).sort_values("_dist_m")
        return pd.Series({
            out_col: df["_station"].tolist(),
            out_dist_col: [round(x, 2) for x in df["_dist_m"].tolist()]
        })

    agg = joined.groupby("_pt_id", as_index=False).apply(agg_station_lists).reset_index(drop=True)

    # Join results back to original points
    pts_out = pts.merge(agg, on="_pt_id", how="left")
    pts_out[out_col] = pts_out[out_col].apply(lambda x: x if isinstance(x, list) else [])
    pts_out[out_dist_col] = pts_out[out_dist_col].apply(lambda x: x if isinstance(x, list) else [])

    # Cleanup
    pts_out = pts_out.drop(columns=["_pt_id"])

    return pts_out

In [19]:
poi_gdf = gpd.GeoDataFrame(
    poi_df,
    geometry=gpd.points_from_xy(poi_df["lon"], poi_df["lat"]),
    crs="EPSG:4326"
)

poi_with_lists = assign_station_lists(
    points_gdf=poi_gdf,
    stations_points_4326=stations,
    walksheds_4326=stations_walkshed_4326,
    station_name_col="name",
    out_col="stations_in_radius",
    out_dist_col="stations_in_radius_dist_m"
)

# --- Save as GeoJSON (lists are fine in GeoJSON) ---
poi_with_lists = poi_with_lists.drop(columns=['station_id', 'osm_type', 'osm_key'])
poi_with_lists.to_file("data/pois_with_station_lists.geojson", driver="GeoJSON")

# --- Save as CSV (lists should be stringified) ---
poi_csv = poi_with_lists.copy()
poi_csv["stations_in_radius"] = poi_csv["stations_in_radius"].apply(lambda x: ",".join(x))
poi_csv["stations_in_radius_dist_m"] = poi_csv["stations_in_radius_dist_m"].apply(lambda x: ",".join(map(str, x)))
poi_csv.drop(columns="geometry").to_csv("data/pois_with_station_lists.csv", index=False)
poi_csv.head(20)

,osm_id,name,lat,lon,tags,amenity,shop,leisure,tourism,office,public_transport,healthcare,craft,geometry,stations_in_radius,stations_in_radius_dist_m
0,357772806,First Ward Elementary School,35.228198,-80.833404,"{'amenity': 'school', 'ele': '228', 'gnis:feat...",school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.8334 35.2282),"9th Street,7th Street","220.95,437.32"
1,367908585,Charlotte Fire Department Station 4,35.232018,-80.839129,"{'addr:city': 'Charlotte', 'addr:housenumber':...",fire_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83913 35.23202),9th Street,455.21
2,957284836,6th Street,35.226627,-80.833456,"{'amenity': 'parking', 'name': '6th Street', '...",parking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83346 35.22663),"9th Street,7th Street","368.59,424.11"
3,4001585783,Aroma,35.229878,-80.839511,"{'addr:city': 'Charlotte', 'addr:housenumber':...",restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83951 35.22988),"7th Street,9th Street","323.04,411.0"
4,4001610982,NaN,35.229738,-80.839606,"{'amenity': 'bicycle_parking', 'bicycle_parkin...",bicycle_parking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83961 35.22974),"7th Street,9th Street","312.71,419.16"
5,4001610992,NaN,35.229504,-80.840018,{'amenity': 'waste_basket'},waste_basket,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.84002 35.2295),"7th Street,9th Street","309.7,457.11"
6,6532532810,7th Restaurant & Lounge,35.226134,-80.836090,"{'addr:city': 'Charlotte', 'addr:housenumber':...",restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83609 35.22613),"7th Street,9th Street","215.98,407.71"
7,9267613529,NaN,35.227393,-80.836323,"{'amenity': 'bench', 'armrest': 'yes', 'backre...",bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83632 35.22739),"7th Street,9th Street","159.24,282.74"
8,9267613530,NaN,35.227585,-80.836069,"{'amenity': 'bench', 'armrest': 'yes', 'backre...",bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83607 35.22758),"7th Street,9th Street","185.68,253.92"
9,9267613531,NaN,35.227817,-80.835974,"{'amenity': 'bench', 'armrest': 'yes', 'backre...",bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83597 35.22782),"7th Street,9th Street","201.07,226.82"


In [22]:
crime_gdf = gpd.GeoDataFrame(
    crime_df,
    geometry=gpd.points_from_xy(crime_df["lon"], crime_df["lat"]),
    crs="EPSG:4326"
)

crime_with_lists = assign_station_lists(
    points_gdf=crime_gdf,
    stations_points_4326=stations,
    walksheds_4326=stations_walkshed_4326,
    station_name_col="name",
    out_col="stations_in_radius",
    out_dist_col="stations_in_radius_dist_m"
)

# Save GeoJSON
crime_with_lists = crime_with_lists.rename(columns={'name': 'nearest_station'})
crime_with_lists.to_file("data/crimes_with_station_lists.geojson", driver="GeoJSON")

# Save CSVs (stringify list columns)
def stringify_lists_for_csv(df):
    df = df.copy()
    df["stations_in_radius"] = df["stations_in_radius"].apply(lambda x: ",".join(x))
    df["stations_in_radius_dist_m"] = df["stations_in_radius_dist_m"].apply(lambda x: ",".join(map(str, x)))
    return df.drop(columns="geometry")

stringify_lists_for_csv(crime_with_lists).to_csv("data/crimes_with_station_lists.csv", index=False)
crime_with_lists.head(20)

,INCIDENT_REPORT_ID,LOCATION,ZIP,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,CMPD_PATROL_DIVISION,NPA,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,...,precip,precipcover,cloudcover,humidity,windspeed,visibility,snow,conditions,stations_in_radius,stations_in_radius_dist_m
0,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,0.011,8.33,53.2,76.0,14.5,9.4,0.0,"Rain, Partially cloudy","[3rd St/Convention, CTC/Arena, Brooklyn Village]","[98.25, 235.05, 461.36]"
1,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,1.307,50.00,89.5,93.3,12.8,8.3,0.0,"Rain, Partially cloudy","[3rd St/Convention, Brooklyn Village]","[335.32, 394.77]"
2,20180916-0854-01,900 SPINDLE ST,28206,35.23,-80.83,Central,476,Indoors,Residential,Private Residence,...,3.875,100.00,99.5,97.1,25.5,3.5,0.0,"Rain, Overcast",[9th Street],[337.83]
3,20180523-1548-04,600 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Open Area,Construction Site,...,0.000,0.00,74.8,76.5,10.0,9.9,0.0,Partially cloudy,[Brooklyn Village],[270.46]
4,20191125-1133-02,200 E BLAND ST,28203,35.22,-80.85,Central,3,Other,Commercial Place,Bar/Tavern/Nightclub,...,0.000,0.00,4.8,69.5,6.5,9.8,0.0,Clear,[Bland],[158.91]
5,20230428-1602-03,600 N CHURCH ST,28202,35.23,-80.84,Central,476,Parking Deck,Residential,Apartment/Duplex Private Res,...,0.844,25.00,78.8,77.1,14.2,9.1,0.0,"Rain, Partially cloudy",[9th Street],[409.9]
6,20240703-0652-00,500 N DAVIDSON ST,28202,35.23,-80.83,Central,476,Indoors,Residential,Apartment/Duplex Private Res,...,0.000,0.00,59.8,58.3,6.3,9.9,0.0,Partially cloudy,[9th Street],[381.8]
7,20260118-0023-00,500 NORTH ELLIS LN,28202,35.23,-80.84,Central,476,Indoors,Commercial Place,Restaurant/Diner/Coffee Shop,...,0.268,54.17,47.9,83.0,12.2,8.2,0.2,"Rain, Partially cloudy","[7th Street, 9th Street]","[190.1, 202.95]"
8,20240229-1100-01,1300 SOUTH BV,28203,35.22,-80.85,Central,3,Indoors,Commercial Place,Other - Commercial Place,...,0.000,0.00,58.3,32.2,12.3,9.9,0.0,Partially cloudy,[Bland],[294.28]
9,20170215-2050-03,300 S COLLEGE ST,28202,35.22,-80.84,Central,476,Indoors,Commercial Place,Restaurant/Diner/Coffee Shop,...,0.422,16.67,54.6,54.3,15.5,9.3,0.0,"Rain, Partially cloudy","[3rd St/Convention, CTC/Arena, Brooklyn Village]","[160.41, 258.28, 444.28]"


In [ ]:
crime_with_lists['lon'].value_counts()

lon
-80.841973    10068
-80.838020     4626
-80.843083     3352
-80.841946     2192
-80.838701     2052
              ...  
-80.831897        3
-80.864722        3
-80.852839        2
-80.849980        2
-80.850257        1
Name: count, Length: 193, dtype: int64